# **2.1 Data Preparation**

The objective of this section is to transform the raw monthly trading data into a format that can be used efficiently for model fitting and backtesting.

The raw bin files are stored in long format, with one row per stock, date and intraday time bin. For price impact modelling, it is more convenient to work with matrices where each row corresponds to one stock-day and each column corresponds to one intraday time bin.

The baseline project setup uses one month as the in-sample training period and the following month as the out-of-sample testing period. The same set of 20 stocks is used in both periods.


Let

$$
q_{i,d,t}
$$

denote the signed traded volume of stock \(i\) on date \(d\) during intraday bin \(t\), and let

$$
P_{i,d,t}
$$

denote the mid price at the end of the same bin.

The aim is to construct two matrices:

$$
Q_{(i,d),t} = q_{i,d,t},
$$

and

$$
P_{(i,d),t} = P_{i,d,t}.
$$

Here, the row index \((i,d)\) represents one stock-day, while the columns represent intraday time bins.

We use January 2019 as the in-sample period and February 2019 as the out-of-sample period.

The in-sample data is used for stock selection and model fitting. The out-of-sample data is only used later to evaluate the fitted model.

This avoids look-ahead bias: the stock universe is selected using only the training month, not the testing month.

The 20-stock universe is selected by liquidity. For each stock \(i\), we compute total absolute traded volume in the training month:

$$
V_i = \sum_{d \in \text{train}} \sum_t |q_{i,d,t}|.
$$

The selected universe is then

$$
\mathcal{S}_{20}
=
\text{top 20 stocks ranked by } V_i.
$$

The same stock set $\mathcal{S}_{20}$ is then used for both the training month and the testing month.

The bin files contain both signed volume and mid prices.

For signed trading volume, we use the column `trade`:

$$
Q_{(i,d),t} = \text{trade}_{i,d,t}.
$$

For prices, we use the column `midEnd`:

$$
P_{(i,d),t} = \text{midEnd}_{i,d,t}.
$$

We convert the raw long data into wide stock-day matrices:

- rows: `(stock, date)`;
- columns: `time`;
- values: either `trade` or `midEnd`.

Missing values are treated differently for trades and prices.

For traded volume, missing values are filled with zero:

$$
q_{i,d,t} = 0
$$

when there is no recorded trade in that bin.

For prices, missing values are forward-filled and backward-filled across time because the absence of a new price observation does not mean that the price is zero. The last available mid price is carried forward.

The output of the data preparation step is:

$$
Q^{\text{train}}, \quad P^{\text{train}}, \quad Q^{\text{test}}, \quad P^{\text{test}}.
$$

These matrices are saved as intermediate results and will be used in the next section to fit price impact models.

This completes the baseline data preparation step.

In [24]:
import os
import importlib
import src.data_prep

importlib.reload(src.data_prep)

from src.data_prep import *

data_dir = "data/"

bin_sample_path = f"{data_dir}binSamples/"
fill_sample_path = f"{data_dir}fillSamples/"

print(os.listdir(bin_sample_path))
print(os.listdir(fill_sample_path))

['bin201901.csv', 'bin201902.csv']
['fills201901.csv', 'fills201902.csv']


In [13]:
# Reading data
# January 2019 = in-sample
# February 2019 = out-of-sample

year = 2019
train_month = 1
test_month = 2

train_bin_df = load_bin_month(bin_sample_path, year, train_month)
test_bin_df = load_bin_month(bin_sample_path, year, test_month)


In [ ]:
# Choosing the 20 most liquid stocks
stocks_20 = select_top_liquid_stocks(train_bin_df, n_stocks=20)

In [ ]:
# Saving the stocks
result_path = "data/"
os.makedirs(result_path, exist_ok=True)

pd.DataFrame({"stock": stocks_20}).to_csv(
    result_path + "stocks_20_201901.csv",
    index=False
)

In [26]:
# Filtering train and test to the same 20 stocks
train_bin_df = train_bin_df.loc[train_bin_df["stock"].isin(stocks_20)].copy()
test_bin_df = test_bin_df.loc[test_bin_df["stock"].isin(stocks_20)].copy()


train_bin_df = train_bin_df.sort_values(["stock", "date", "time"])
test_bin_df = test_bin_df.sort_values(["stock", "date", "time"])

In [27]:
# Build stock-date x time matrices
train_traded_volume_df = make_panel(train_bin_df, "trade", "zero")
test_traded_volume_df = make_panel(test_bin_df, "trade", "zero")
train_px_df = make_panel(train_bin_df, "midEnd", "price")
test_px_df = make_panel(test_bin_df, "midEnd", "price")

In [28]:
# Align train and test time columns
(
    train_traded_volume_df,
    test_traded_volume_df,
    train_px_df,
    test_px_df
) = align_intraday_columns(
    train_traded_volume_df,
    test_traded_volume_df,
    train_px_df,
    test_px_df
)

In [ ]:
train_traded_volume_df.head(2)

time              09:30:00  09:30:10  09:30:20  09:30:30  09:30:40  09:30:50  \
stock date                                                                     
AAL   2019-01-02    31.525     31.52    31.535    31.515    31.485    31.435   
      2019-01-03    31.770     31.80    31.840    31.765    31.740    31.720   

time              09:31:00  09:31:10  09:31:20  09:31:30  ...  15:58:30  \
stock date                                                ...             
AAL   2019-01-02    31.445    31.480    31.480     31.44  ...    32.455   
      2019-01-03    31.660    31.665    31.545     31.53  ...    30.085   

time              15:58:40  15:58:50  15:59:00  15:59:10  15:59:20  15:59:30  \
stock date                                                                     
AAL   2019-01-02    32.445    32.445    32.445    32.465    32.465    32.480   
      2019-01-03    30.100    30.115    30.120    30.125    30.115    30.095   

time              15:59:40  15:59:50  16:00:00  
stock date                                      
AAL   2019-01-02    32.500    32.475    32.475  
      2019-01-03    30.075    30.015    30.015  

[2 rows x 2341 columns]

In [ ]:
# Compute scaling variables: ADV and volatility
daily_volume = train_traded_volume_df.abs().sum(axis=1)

stock_adv = (
    daily_volume
    .groupby(level="stock")
    .mean()
    .rename("ADV")
)

daily_return = train_px_df.iloc[:, -1] / train_px_df.iloc[:, 0] - 1 
# (last price of the day / first price of the day) - 1

stock_vol = (
    daily_return
    .groupby(level="stock")
    .std()
    .rename("sigma")
)

scaling_df = pd.concat([stock_adv, stock_vol], axis=1)

In [38]:
scaling_df.head(2)

,ADV,sigma
stock,,
AAL,1.236722e+06,0.031453
AAPL,2.968463e+06,0.011968


In [39]:
# 8. Diagnostics
print("Selected stocks:")
print(stocks_20)

print("\nShapes:")
print("train_traded_volume_df:", train_traded_volume_df.shape)
print("test_traded_volume_df :", test_traded_volume_df.shape)
print("train_px_df           :", train_px_df.shape)
print("test_px_df            :", test_px_df.shape)
print("scaling_df            :", scaling_df.shape)

print("\nMissing values:")
print("train_traded_volume_df:", train_traded_volume_df.isna().sum().sum())
print("test_traded_volume_df :", test_traded_volume_df.isna().sum().sum())
print("train_px_df           :", train_px_df.isna().sum().sum())
print("test_px_df            :", test_px_df.isna().sum().sum())

print("\nTotal absolute traded volume:")
print("Train:", train_traded_volume_df.abs().sum().sum())
print("Test :", test_traded_volume_df.abs().sum().sum())

print("\nScaling variables:")
display(scaling_df.head())


Selected stocks:
['AMD', 'AAPL', 'AMAT', 'AAL', 'AMZN', 'AMGN', 'ABT', 'ABBV', 'APA', 'ADI', 'ADBE', 'APC', 'AES', 'AIG', 'AFL', 'ADP', 'AEP', 'ALXN', 'ADM', 'ALGN']

Shapes:
train_traded_volume_df: (420, 2341)
test_traded_volume_df : (380, 2341)
train_px_df           : (420, 2341)
test_px_df            : (380, 2341)
scaling_df            : (20, 2)

Missing values:
train_traded_volume_df: 0
test_traded_volume_df : 0
train_px_df           : 0
test_px_df            : 0

Total absolute traded volume:
Train: 442042694.0
Test : 325471405.0

Scaling variables:


,ADV,sigma
stock,,
AAL,1.236722e+06,0.031453
AAPL,2.968463e+06,0.011968
ABBV,4.763432e+05,0.017574
ABT,5.000601e+05,0.017122
ADBE,4.143660e+05,0.013295


In [41]:
# Saving the files for the 20 stocks
train_traded_volume_df.reset_index().to_csv(
    result_path + "train_traded_volume_201901_20.csv",
    index=False
)

train_px_df.reset_index().to_csv(
    result_path + "train_px_201901_20.csv",
    index=False
)

test_traded_volume_df.reset_index().to_csv(
    result_path + "test_traded_volume_201902_20.csv",
    index=False
)

test_px_df.reset_index().to_csv(
    result_path + "test_px_201902_20.csv",
    index=False
)

scaling_df.reset_index().to_csv(
    result_path + "scaling_201901_20.csv",
    index=False
)